# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [118]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [119]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [120]:
#your code here

features = spaceship.drop(columns='Transported')
target =spaceship['Transported']
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size = 0.20, random_state=10)

In [121]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy="mean")
num_cols = X_train.select_dtypes(include="number").columns

In [122]:
num_cols

Index(['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck'], dtype='object')

In [123]:
imputer.fit(X_train[num_cols])
X_train[num_cols] = imputer.transform(X_train[num_cols])
X_test[num_cols] = imputer.transform(X_test[num_cols])

In [124]:
imputer =  SimpleImputer(strategy="most_frequent")
cat_cols=["HomePlanet", "CryoSleep", "Cabin", "Destination", "VIP"]
imputer.fit(X_train[cat_cols])
X_train[cat_cols] = imputer.transform(X_train[cat_cols])
X_test[cat_cols] = imputer.transform(X_test[cat_cols])

In [125]:
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder(sparse_output= False ,handle_unknown= 'ignore')

X_train["Cabin"] = X_train["Cabin"].str[0]
X_test["Cabin"] = X_test["Cabin"].str[0]

In [126]:
X_train = X_train.drop(columns=['PassengerId','Name'])
X_test = X_test.drop(columns=['PassengerId','Name'])

encoder.fit(X_train[cat_cols])
encoded_train= encoder.transform(X_train[cat_cols])
encoded_test= encoder.transform(X_test[cat_cols])

In [127]:
encoded_df_train = pd.DataFrame(
    encoded_train,
    columns=encoder.get_feature_names_out(cat_cols),
    index=X_train.index
)

X_train_final = pd.concat([X_train.drop(columns=cat_cols), encoded_df_train], axis=1)
encoded_df_test = pd.DataFrame(
    encoded_test,
    columns=encoder.get_feature_names_out(cat_cols),
    index=X_test.index
)

X_test_final = pd.concat([X_test.drop(columns=cat_cols), encoded_df_test], axis=1)

**Perform Train Test Split**

**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

In [128]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=11)
knn.fit(X_train_final, y_train)
pred = knn.predict(X_test_final)
from sklearn.metrics import accuracy_score, precision_score, recall_score
print("Accuracy:", accuracy_score(y_test, pred))
print("Precision:", precision_score(y_test, pred))
print("Recall:", recall_score(y_test, pred))

Accuracy: 0.7768832662449684
Precision: 0.7629213483146068
Recall: 0.7932242990654206


In [129]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
scaler.fit(X_train_final)                          # learn min/max from TRAIN only
X_train_scaled = scaler.transform(X_train_final)
X_test_scaled = scaler.transform(X_test_final)

knn_scaled = KNeighborsClassifier(n_neighbors=11)
knn_scaled.fit(X_train_scaled, y_train)
pred_scaled = knn_scaled.predict(X_test_scaled)

print("Accuracy (scaled):", accuracy_score(y_test, pred_scaled))
print(classification_report(y_test, pred_scaled))

Accuracy (scaled): 0.7694077055779184
              precision    recall  f1-score   support

       False       0.77      0.78      0.77       883
        True       0.77      0.76      0.76       856

    accuracy                           0.77      1739
   macro avg       0.77      0.77      0.77      1739
weighted avg       0.77      0.77      0.77      1739



- Bagging and Pasting

In [130]:



from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

In [131]:
#your code here
bagging_clf = BaggingClassifier(DecisionTreeClassifier(max_depth=10),
                                n_estimators=100,
                                max_samples=1000,
                                random_state=42)

In [132]:
bagging_clf.fit(X_train_final, y_train)

,estimator,DecisionTreeC...(max_depth=10)
,n_estimators,100
,max_samples,1000
,max_features,1.0
,bootstrap,True
,bootstrap_features,False
,oob_score,False
,warm_start,False
,n_jobs,None
,random_state,42
,verbose,0


In [133]:
pred = bagging_clf.predict(X_test_final)
print(classification_report(y_test, pred))


              precision    recall  f1-score   support

       False       0.82      0.78      0.80       883
        True       0.78      0.82      0.80       856

    accuracy                           0.80      1739
   macro avg       0.80      0.80      0.80      1739
weighted avg       0.80      0.80      0.80      1739



In [134]:
from sklearn.metrics import classification_report

print(classification_report(y_test, pred))

              precision    recall  f1-score   support

       False       0.82      0.78      0.80       883
        True       0.78      0.82      0.80       856

    accuracy                           0.80      1739
   macro avg       0.80      0.80      0.80      1739
weighted avg       0.80      0.80      0.80      1739



- Random Forests

In [135]:
#your code here
from sklearn.ensemble import RandomForestClassifier
randomforest_clf = RandomForestClassifier(max_depth=10,n_estimators=100, max_samples=1000,random_state=42)

In [136]:
randomforest_clf.fit(X_train_final, y_train)
pred = randomforest_clf.predict(X_test_final)
accuracy_score(y_test,pred)

0.7993099482461185

In [137]:

print(classification_report(y_test, pred))

              precision    recall  f1-score   support

       False       0.82      0.78      0.80       883
        True       0.78      0.82      0.80       856

    accuracy                           0.80      1739
   macro avg       0.80      0.80      0.80      1739
weighted avg       0.80      0.80      0.80      1739



- Gradient Boosting

In [138]:
#your code here
from sklearn.ensemble import GradientBoostingClassifier
gradient_clf = GradientBoostingClassifier(max_depth=3,n_estimators=100, random_state=42)

In [139]:
gradient_clf.fit(X_train_final, y_train)
pred = gradient_clf.predict(X_test_final)
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

       False       0.84      0.74      0.79       883
        True       0.76      0.85      0.80       856

    accuracy                           0.80      1739
   macro avg       0.80      0.80      0.80      1739
weighted avg       0.80      0.80      0.80      1739



- Adaptive Boosting

In [140]:
#your code here
from sklearn.ensemble import AdaBoostClassifier
adaboost_clf = AdaBoostClassifier(n_estimators=100, random_state=42)

In [141]:
adaboost_clf.fit(X_train_final, y_train)
pred = adaboost_clf.predict(X_test_final)
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

       False       0.78      0.79      0.78       883
        True       0.78      0.77      0.77       856

    accuracy                           0.78      1739
   macro avg       0.78      0.78      0.78      1739
weighted avg       0.78      0.78      0.78      1739



Which model is the best and why?

In [ ]:
#Random Forest, Bagging and Gradient Boosting all reached about 0.80 accuracy, which is better than KNN (0.77–0.78).
#  I would choose Random Forest because it had the highest accuracy and the most balanced precision and recall for both classes. 
# Gradient Boosting had similar accuracy but was less balanced (recall 0.74 for False vs. 0.85 for True).
